# 🧠 Backpropagation
**Course:** Deep Learning  
**Lecture:** 2 — MLP, Backpropagation, and Activation Functions  
**Part:** b — Backpropagation

---

> **Where we are:** Now that we can pass data forward to get predictions, we need to pass errors backward to update the weights.

**What you'll learn:**
- The chain rule applied to neural networks
- Implementing backprop by hand in NumPy
- Verifying gradients with TensorFlow's `GradientTape`
- The vanishing gradient problem (empirical visualization)
- Numerical gradient checking

**Exam relevance:** ⭐⭐⭐ (You must know the chain rule derivations and the vanishing gradient concept inside-out.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 📖 1. The Chain Rule & Derivatives
To update a weight $W$, we need to know how much the loss $L$ changes when $W$ changes. This is $\frac{\partial L}{\partial W}$.

By the chain rule:
$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial W}$$

For a Dense Layer $z = XW + b$, $a = f(z)$:
1. $\frac{\partial z}{\partial W} = X^T$
2. $\frac{\partial a}{\partial z} = f'(z)$
3. $\frac{\partial L}{\partial a}$ comes from the next layer (or the loss function if it's the last layer).

## 🔀 Your Options: Loss Function
| Option | Pros | Cons | When to use |
|---|---|---|---|
| **MSE (Mean Squared Error)** | Easy derivative, symmetric | Sensitive to outliers | Regression tasks |
| **MAE (Mean Absolute Error)** | Robust to outliers | Derivative is discontinuous at 0 | Regression with noisy labels |
| **Cross-Entropy** | Punishes confident wrong answers strongly | Requires probabilities | Classification tasks |

In [ ]:
# ✏️ YOUR TURN
# Implement MSE loss and its gradient
def mse_loss_student(y_true, y_pred):
    ### YOUR CODE HERE ###
    # return np.mean(...)
    pass

def mse_gradient_student(y_true, y_pred):
    ### YOUR CODE HERE ###
    # Remember: derivative of mean((y_pred - y_true)^2)
    # return 2 * (...) / N
    pass

In [ ]:
# ✅ SOLUTION
def mse_loss(y_true, y_pred):
    return np.mean((y_pred - y_true)**2)

def mse_gradient(y_true, y_pred):
    N = y_true.shape[0]
    return 2.0 * (y_pred - y_true) / N

## 📖 2. Manual Backpropagation (Single Layer)
Let's implement a single layer forward and backward pass.

In [ ]:
# Setup data
X = np.array([[1.0, 2.0]])
y_true = np.array([[0.5]])

W = np.array([[0.1], [0.2]])
b = np.array([[0.0]])

# ✏️ YOUR TURN
# Forward pass (Linear activation, so z = a)
### YOUR CODE HERE ###
# y_pred = ...

# Loss
# loss = ...

# Backward pass
# dL_dy = ...
# dL_dW = ... # Hint: X.T @ dL_dy
# dL_db = ... # Hint: np.sum(dL_dy, axis=0, keepdims=True)

In [ ]:
# ✅ SOLUTION
y_pred = X @ W + b
loss = mse_loss(y_true, y_pred)

dL_dy = mse_gradient(y_true, y_pred)
dL_dW = X.T @ dL_dy
dL_db = np.sum(dL_dy, axis=0, keepdims=True)

print("NumPy Gradients:")
print("dL_dW:\n", dL_dW)
print("dL_db:\n", dL_db)

## 🔬 3. Verify with TensorFlow
We will run the exact same computation using `tf.GradientTape`.

In [ ]:
tf_X = tf.constant(X)
tf_y = tf.constant(y_true)
tf_W = tf.Variable(W)
tf_b = tf.Variable(b)

with tf.GradientTape() as tape:
    tf_y_pred = tf.matmul(tf_X, tf_W) + tf_b
    tf_loss = tf.reduce_mean(tf.square(tf_y_pred - tf_y))

tf_grads = tape.gradient(tf_loss, [tf_W, tf_b])

print("TensorFlow Gradients:")
print("dL_dW:\n", tf_grads[0].numpy())
print("dL_db:\n", tf_grads[1].numpy())

np.testing.assert_allclose(dL_dW, tf_grads[0].numpy(), atol=1e-6)
np.testing.assert_allclose(dL_db, tf_grads[1].numpy(), atol=1e-6)
print("\n\u2705 Gradients match perfectly!")

## 📖 4. Numerical Gradient Checking
Before AutoDiff frameworks, we had to verify our analytical gradients via finite differences:
$$f'(w) \approx \frac{f(w+\epsilon) - f(w-\epsilon)}{2\epsilon}$$

In [ ]:
epsilon = 1e-4

# ✏️ YOUR TURN
# Modify W[0, 0] by +epsilon and -epsilon, compute loss for both, and find the gradient
### YOUR CODE HERE ###
# W_plus = ...
# W_minus = ...
# loss_plus = ...
# loss_minus = ...
# num_grad_w0 = ...


In [ ]:
# ✅ SOLUTION
W_plus = W.copy()
W_plus[0, 0] += epsilon
loss_plus = mse_loss(y_true, X @ W_plus + b)

W_minus = W.copy()
W_minus[0, 0] -= epsilon
loss_minus = mse_loss(y_true, X @ W_minus + b)

num_grad_w00 = (loss_plus - loss_minus) / (2 * epsilon)

print(f"Analytical dL/dW[0,0]: {dL_dW[0, 0]:.6f}")
print(f"Numerical  dL/dW[0,0]: {num_grad_w00:.6f}")

## 🔬 5. Vanishing Gradient Problem
Let's build a deep 10-layer network and plot the gradient magnitude at each layer for Sigmoid vs ReLU.

In [ ]:
def build_model(activation):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(32, activation=activation, input_shape=(10,)))
    for _ in range(9):
        model.add(tf.keras.layers.Dense(32, activation=activation))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    return model

X_deep = tf.random.normal((32, 10))
y_deep = tf.random.normal((32, 1))

def get_gradients(model):
    with tf.GradientTape() as tape:
        preds = model(X_deep)
        loss = tf.reduce_mean(tf.square(preds - y_deep))
    # Get gradients of loss w.r.t all trainable variables (weights only, skipping biases for plot)
    weights = [var for var in model.trainable_variables if 'bias' not in var.name]
    grads = tape.gradient(loss, weights)
    # Compute mean absolute gradient for each layer
    return [np.mean(np.abs(g.numpy())) for g in grads]

model_sig = build_model('sigmoid')
model_relu = build_model('relu')

grads_sig = get_gradients(model_sig)
grads_relu = get_gradients(model_relu)

# The lists are from input to output. Let's plot them.
plt.figure(figsize=(10, 5))
plt.plot(range(1, 12), grads_sig, marker='o', label='Sigmoid (Vanishing)')
plt.plot(range(1, 12), grads_relu, marker='s', label='ReLU (Healthy)')
plt.yscale('log')
plt.xlabel('Layer Number (1=First Hidden, 11=Output)')
plt.ylabel('Mean Absolute Gradient (Log Scale)')
plt.title('Vanishing Gradient Problem in Deep Networks')
plt.legend()
plt.show()

## 🎓 Exam Corner

### Question 1 (Conceptual)
Write the chain rule for $\frac{\partial L}{\partial W_1}$ in a 2-layer network $y = f(W_2 f(W_1 x))$.

<details><summary>💡 Answer</summary>

Let $z_1 = W_1 x$, $a_1 = f(z_1)$, $z_2 = W_2 a_1$, $y = f(z_2)$.
$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial z_2} \cdot \frac{\partial z_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial W_1}$

</details>

### Question 2 (Conceptual)
What is the vanishing gradient problem and which activation functions suffer from it?

<details><summary>💡 Answer</summary>

The vanishing gradient problem occurs when backpropagating through many layers. Because the chain rule multiplies derivatives, if the local derivatives are strictly less than 1, the gradient exponentially decays to 0 in earlier layers.
**Sigmoid** and **Tanh** suffer from this because their max derivatives are 0.25 and 1.0 respectively, and they saturate (derivative approaches 0) for large inputs.

</details>

### Question 3 (Conceptual)
Why does ReLU help with vanishing gradients?

<details><summary>💡 Answer</summary>

For positive inputs, the derivative of ReLU is exactly 1. When applying the chain rule, multiplying by 1 does not shrink the gradient, allowing it to propagate back through many layers without vanishing.

</details>